In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import requests
import pandas as pd

def searxng_test(query):
    url = "http://localhost:8080/search"
    params = {
        'q': query,
        'format': 'json',
        'language': 'ko-KR'
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        results = response.json().get('results', [])
        
        # 데이터 가공 (제목, URL, 출처)
        data = []
        for res in results:
            data.append({
                "Title": res.get('title'),
                "URL": res.get('url'),
                "Source": res.get('engine'),
                "Snippet": res.get('content', '')[:100] + "..."
            })
        
        # Pandas 데이터프레임으로 변환하여 표 형식으로 출력
        df = pd.DataFrame(data)
        return df

    except Exception as e:
        print(f"Error: {e}")
        return None

# 실행 테스트
query = "Vietnam baby food market trend 2026"
df_results = searxng_test(query)

# 결과가 있으면 출력
if df_results is not None:
    display(df_results.head(100)) # 상위 10개 결과 출력

,Title,URL,Source,Snippet
0,What are the leading brands in the Vietnam Bab...,https://www.kenresearch.com/vietnam-baby-food-...,brave,The Vietnam Baby Food and Infant Care Market i...
1,Vietnam Baby Food Market Size and Share by Cat...,https://www.globaldata.com/store/report/vietna...,brave,Market profile of the various product sectors ...
2,Baby Products Statistics and Facts (2026),https://www.news.market.us/baby-products-stati...,brave,Vietnam and Russia also demonstrated significa...
3,Vietnam's Baby Food Market Report 2025 - Price...,https://www.indexbox.io/store/vietnam-food-pre...,brave,This report provides an in-depth analysis of t...
4,Baby Food - Vietnam | Statista Market Forecast,https://www.statista.com/outlook/emo/food/baby...,brave,"Trends in the market: In Vietnam, the baby foo..."
5,Baby Food in Vietnam | Market Research Report ...,https://www.euromonitor.com/baby-food-in-vietn...,brave,...
6,Vietnam Baby Food Market (2021 - 2031) | Trend...,https://www.6wresearch.com/industry-report/vie...,brave,...
7,[베트남] 영유아식품 시장 동향,https://m.kati.net/board/exportNewsView.do?boa...,google,2025. 12. 31. — 베트남 영유아식품 시장 동향. - 유로모니터(Eurom...
8,"Vietnam Baby Food Market Share, Forecasts, Str...",https://www.linkedin.com/pulse/vietnam-baby-fo...,duckduckgo,The Vietnam Baby Food Market is witnessing acc...
9,[베트남] 영유아식품 시장 동향 - 곡산 - 티스토리,https://why4416.tistory.com/m/15988820,google,2026. 1. 3. — - Vietnam Mother & Baby Market 2...


In [22]:
import os
import requests
import pandas as pd
from tavily import TavilyClient
from dotenv import load_dotenv

# 1. .env 파일로부터 환경 변수 로드
load_dotenv()

# 환경 변수 가져오기
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
SEARXNG_URL = os.getenv("SEARXNG_URL")

# Tavily 클라이언트 초기화
tavily = TavilyClient(api_key=TAVILY_API_KEY)

def market_insight_search(query, max_results=10):
    print(f"🚀 프로젝트 [베트남 건조이유식] 관련 데이터 수집 시작: {query}")
    
    # --- STEP 1: SearXNG 검색 ---
    params = {
        'q': query,
        'format': 'json',
        'language': 'en-US',
        'time_range': 'year'
    }
    
    try:
        response = requests.get(SEARXNG_URL, params=params)
        search_results = response.json().get('results', [])
    except Exception as e:
        print(f"❌ SearXNG 접속 실패: {e}")
        return None

    # --- STEP 2: 분석 및 데이터 가공 ---
    final_data = []
    
    for i, res in enumerate(search_results[:max_results]):
        url = res.get('url')
        title = res.get('title')
        
        info = {
            "Rank": i + 1,
            "Title": title,
            "URL": url,
            "Source": res.get('engine', 'N/A'),
            "Snippet": res.get('content', ''),
            "Key_Insight": "N/A"
        }
        
        # 핵심 3개 아티클은 Tavily로 본문 요약 (마케팅 인사이트 도출용)
        if i < 3:
            try:
                # Tavily Search API의 search_depth='advanced'를 활용해 요약까지 한 번에 가져오기
                # URL 직접 추출이 목적이라면 .extract()를, 검색과 분석을 동시에 하려면 .search()를 씁니다.
                context = tavily.search(query=query, search_depth="advanced", max_results=1)
                info["Key_Insight"] = context['results'][0].get('content', '분석 중...')
            except:
                info["Key_Insight"] = "인사이트 추출 실패"

        final_data.append(info)

    return pd.DataFrame(final_data)

# --- 3. 실전 실행 및 대시보드 데이터 생성 ---
query = "Vietnam baby food market logistics and climate challenges for dry food"
df_dashboard = market_insight_search(query)

if df_dashboard is not None:
    # 대시보드용 CSV 저장
    df_dashboard.to_csv("vietnam_campaign_data.csv", index=False, encoding="utf-8-sig")
    print("\n✅ 대시보드용 데이터 소스 생성 완료!")
    display(df_dashboard[['Rank', 'Title', 'URL', 'Source']].head())

🚀 프로젝트 [베트남 건조이유식] 관련 데이터 수집 시작: Vietnam baby food market logistics and climate challenges for dry food

✅ 대시보드용 데이터 소스 생성 완료!


,Rank,Title,URL,Source
0,1,Vietnam Baby Food & Infant Formula Market,https://www.kenresearch.com/vietnam-baby-food-...,google
1,2,Vietnam Organic Baby Food Market 2026 | Strate...,https://www.linkedin.com/pulse/vietnam-organic...,google
2,3,Vietnam Baby Food and Infant Care Market,https://www.kenresearch.com/vietnam-baby-food-...,google
3,4,"Baby Food Market Size, Share, Analysis, & Tren...",https://www.marketdataforecast.com/market-repo...,google
4,5,Tracking a decade of food system transformatio...,https://www.frontiersin.org/journals/sustainab...,google
